# CareAssist — Ensemble Placement Stability Prediction

**Combines 3 models trained on real AFCARS FY2020–2024 data (5 years)**

| Model | Individual AUC |
|-------|---------------|
| XGBoost | ~0.906 |
| LightGBM | ~0.901 |
| CatBoost | ~0.899 |

### Ensemble Methods
1. **Simple Average** — equal-weight probability averaging
2. **Optimized Weighted Average** — grid-search best weights
3. **Rank Average** — average percentile ranks (robust to calibration differences)
4. **Stacking** — Logistic Regression meta-learner on base model outputs

### Instructions
1. Run Cell 1 to install packages
2. Run Cell 2 (imports)
3. Run Cell 3 — upload `Berkeley_AFCARS_CSVs_UTF8.zip`
4. Then **Runtime > Run all remaining cells**
5. Cell 21 will auto-download a zip with all outputs

In [ ]:
# Cell 1: Install packages
!pip install -q xgboost lightgbm catboost shap imbalanced-learn

In [ ]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, os, glob, zipfile, shutil, itertools
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve, f1_score
)
from scipy.stats import rankdata

import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import shap
import joblib

plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
print('All imports OK')

In [ ]:
# Cell 3: Upload Berkeley_AFCARS_CSVs_UTF8.zip
from google.colab import files
print('Upload Berkeley_AFCARS_CSVs_UTF8.zip')
uploaded = files.upload()
print(f'Uploaded {len(uploaded)} file(s)')

for zf in uploaded.keys():
    if zf.endswith('.zip'):
        print(f'Extracting {zf}...')
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall('afcars_csv')
            print(f'  Extracted {len(z.namelist())} files')
print('Done')

In [ ]:
# Cell 4: Load CSV files (FY2020-2024, deduplicated)
csv_files = sorted(set(
    glob.glob('afcars_csv/**/*.csv', recursive=True) +
    glob.glob('afcars_csv/*.csv')
))
if not csv_files:
    csv_files = sorted(set(glob.glob('**/*.csv', recursive=True)))
csv_files = [f for f in csv_files if '2019' not in f]
print(f'Found {len(csv_files)} CSV files (2019 excluded):')
for f in csv_files:
    print(f'  {f}')

USE_COLS = [
    'RecNumbr', 'FIPSCode', 'StFCID', 'FY',
    'NUMPLEP', 'TOTALREM', 'CURPLSET', 'CASEGOAL', 'SEX', 'CLINDIS',
    'MR', 'VISHEAR', 'PHYDIS', 'EmotDist', 'OTHERMED', 'CHBEHPRB',
    'PHYABUSE', 'SEXABUSE', 'NEGLECT', 'AAPARENT', 'DAPARENT',
    'AACHILD', 'DACHILD', 'CHILDIS', 'PRTSDIED', 'PRTSJAIL',
    'NOCOPE', 'ABANDMNT', 'RELINQSH', 'HOUSING', 'MANREM',
    'EVERADPT', 'DISREASN', 'PLACEOUT',
    'AgeAtLatRem', 'RaceEthn', 'SettingLOS', 'LatRemLOS',
]

def load_csv(path):
    header = pd.read_csv(path, nrows=0)
    cols = [c for c in USE_COLS if c in header.columns]
    d = pd.read_csv(path, usecols=cols, dtype=str, low_memory=False)
    for c in d.columns:
        if c not in ('RecNumbr', 'FIPSCode', 'StFCID', 'FY'):
            d[c] = pd.to_numeric(d[c], errors='coerce')
    return d

dfs = []
for f in csv_files:
    print(f'Loading {os.path.basename(f)}...')
    d = load_csv(f)
    print(f'  {len(d):,} records')
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
print(f'\nCombined: {len(df):,} records x {df.shape[1]} columns')
print(f'Fiscal years: {sorted(df["FY"].dropna().unique())}')
df.head()

In [ ]:
# Cell 5: Define target variable
# Disrupted = discharge reason Transfer(6)/Runaway(7) OR 3+ placements
# NUMPLEP excluded from features to avoid data leakage

DISRUPTION_CODES = {6, 7}

def label_disruption(row):
    d = row['DISREASN']
    n = row['NUMPLEP']
    if pd.notna(d) and int(d) in DISRUPTION_CODES:
        return 1
    if pd.notna(n) and int(n) >= 3:
        return 1
    return 0

df['disruption'] = df.apply(label_disruption, axis=1)
pos = df['disruption'].sum()
print(f'Disrupted: {pos:,} ({100*pos/len(df):.1f}%)')
print(f'Stable:    {len(df)-pos:,} ({100*(len(df)-pos)/len(df):.1f}%)')
print('\nBy fiscal year:')
print(df.groupby('FY')['disruption'].agg(['count','mean','sum']).to_string())

In [ ]:
# Cell 6: Feature engineering
df['age_at_removal'] = df['AgeAtLatRem'].where(df['AgeAtLatRem'] < 99)

disability_cols = ['MR', 'VISHEAR', 'PHYDIS', 'EmotDist', 'OTHERMED']
for c in disability_cols:
    df[c] = df[c].fillna(0).clip(0, 1).astype(int)
df['has_disability'] = df[disability_cols].max(axis=1)
df['has_clinical_disability'] = (df['CLINDIS'] == 1).astype(int)
df['has_behavioral'] = df['CHBEHPRB'].fillna(0).clip(0, 1).astype(int)

removal_reason_cols = [
    'PHYABUSE','SEXABUSE','NEGLECT','AAPARENT','DAPARENT',
    'AACHILD','DACHILD','CHILDIS','PRTSDIED','PRTSJAIL',
    'NOCOPE','ABANDMNT','RELINQSH','HOUSING'
]
for c in removal_reason_cols:
    df[c] = df[c].fillna(0).clip(0, 1).astype(int)
df['num_removal_reasons'] = df[removal_reason_cols].sum(axis=1)

df['placement_type'] = df['CURPLSET'].where(df['CURPLSET'].isin([1,2,3,4,5,6,7,8]))
df['case_goal'] = df['CASEGOAL'].where(df['CASEGOAL'].isin([1,2,3,4,5,6,7]))
df['total_removals'] = df['TOTALREM'].where(df['TOTALREM'] < 98)

if 'SettingLOS' in df.columns:
    df['los_current_setting'] = pd.to_numeric(df['SettingLOS'], errors='coerce')
else:
    df['los_current_setting'] = np.nan
if 'LatRemLOS' in df.columns:
    df['los_latest_removal'] = pd.to_numeric(df['LatRemLOS'], errors='coerce')
else:
    df['los_latest_removal'] = np.nan

df['ever_adopted'] = df['EVERADPT'].fillna(0).clip(0, 1).astype(int)
df['mandatory_removal'] = df['MANREM'].fillna(0).clip(0, 1).astype(int)

FEATURE_COLS = [
    'age_at_removal',
    'total_removals','placement_type','case_goal',
    'has_disability','has_clinical_disability','has_behavioral',
    'num_removal_reasons',
    'PHYABUSE','SEXABUSE','NEGLECT','AAPARENT','DAPARENT',
    'NOCOPE','ABANDMNT','HOUSING',
    'los_current_setting','los_latest_removal',
    'ever_adopted','mandatory_removal',
]
print(f'{len(FEATURE_COLS)} features (race, gender, NUMPLEP excluded)')

In [ ]:
# Cell 7: EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

df['disruption'].value_counts().plot(kind='bar', ax=axes[0,0], color=['steelblue','coral'])
axes[0,0].set_title('Target Distribution')
axes[0,0].set_xticklabels(['Stable','Disrupted'], rotation=0)

df[df['disruption']==0]['age_at_removal'].hist(bins=20, ax=axes[0,1], alpha=0.6, label='Stable', color='steelblue')
df[df['disruption']==1]['age_at_removal'].hist(bins=20, ax=axes[0,1], alpha=0.6, label='Disrupted', color='coral')
axes[0,1].set_title('Age at Removal'); axes[0,1].legend()

pt_labels = {1:'Pre-Adopt',2:'Foster-Rel',3:'Foster-NonRel',4:'Group Home',5:'Institution',6:'Sup IL',7:'Runaway',8:'Trial Home'}
pt_rates = df.groupby('placement_type')['disruption'].mean().sort_values(ascending=False)
pt_rates.index = [pt_labels.get(int(i),str(i)) for i in pt_rates.index]
pt_rates.plot(kind='barh', ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Disruption by Placement Type')

cg_labels = {1:'Reunify',2:'Relative',3:'Adoption',4:'Long-term FC',5:'Emancipation',6:'Guardianship',7:'Not established'}
cg_rates = df.groupby('case_goal')['disruption'].mean().sort_values(ascending=False)
cg_rates.index = [cg_labels.get(int(i),str(i)) for i in cg_rates.index]
cg_rates.plot(kind='barh', ax=axes[1,0], color='coral')
axes[1,0].set_title('Disruption by Case Goal')

tr = df.groupby('total_removals')['disruption'].mean().head(10)
tr.plot(kind='bar', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Disruption by Total Removals')
axes[1,1].tick_params(axis='x', rotation=0)

yr = df.groupby('FY')['disruption'].mean()
yr.plot(kind='bar', ax=axes[1,2], color='coral')
axes[1,2].set_title('Disruption by Year')
axes[1,2].tick_params(axis='x', rotation=45)

plt.suptitle('CareAssist EDA (FY2020-2024)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8: Prepare data (one-hot encode categoricals)
model_df = df[FEATURE_COLS + ['disruption']].copy()
cat_cols = ['placement_type', 'case_goal']
model_df = pd.get_dummies(model_df, columns=cat_cols, prefix=cat_cols, dummy_na=False)

# Fill remaining NaNs with median
for c in model_df.columns:
    if model_df[c].isna().any():
        model_df[c] = model_df[c].fillna(model_df[c].median())

y = model_df['disruption']
X = model_df.drop(columns=['disruption'])
feature_names = list(X.columns)

print(f'Features: {len(feature_names)}, Records: {len(X):,}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'Disruption rate: train={y_train.mean():.3f} test={y_test.mean():.3f}')
print('Skipping SMOTE - 36% positive rate is balanced enough')

---
## Train Individual Base Models
Each model uses the **same train/test split** (`random_state=42`) so predictions are directly comparable.

In [ ]:
# Cell 9: Train XGBoost
print('=' * 60)
print('Training XGBoost...')
print('=' * 60)
neg, pos_count = (y_train==0).sum(), (y_train==1).sum()
scale_pos = neg / max(pos_count, 1)

xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    tree_method='hist', eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb_model.fit(X_train, y_train, verbose=False)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_ap = average_precision_score(y_test, xgb_proba)
print(f'  ROC-AUC: {xgb_auc:.4f}  Avg Precision: {xgb_ap:.4f}')

In [ ]:
# Cell 10: Train LightGBM
print('=' * 60)
print('Training LightGBM...')
print('=' * 60)

lgbm_model = LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_model.fit(X_train, y_train)

lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]
lgbm_auc = roc_auc_score(y_test, lgbm_proba)
lgbm_ap = average_precision_score(y_test, lgbm_proba)
print(f'  ROC-AUC: {lgbm_auc:.4f}  Avg Precision: {lgbm_ap:.4f}')

In [ ]:
# Cell 11: Train CatBoost
print('=' * 60)
print('Training CatBoost...')
print('=' * 60)

cat_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=0
)
cat_model.fit(X_train, y_train)

cat_proba = cat_model.predict_proba(X_test)[:, 1]
cat_auc = roc_auc_score(y_test, cat_proba)
cat_ap = average_precision_score(y_test, cat_proba)
print(f'  ROC-AUC: {cat_auc:.4f}  Avg Precision: {cat_ap:.4f}')

In [ ]:
# Cell 12: Individual model comparison
print('\n' + '=' * 60)
print('INDIVIDUAL MODEL COMPARISON')
print('=' * 60)

THRESHOLD = 0.4
models_info = {
    'XGBoost':  {'proba': xgb_proba,  'auc': xgb_auc,  'ap': xgb_ap},
    'LightGBM': {'proba': lgbm_proba, 'auc': lgbm_auc, 'ap': lgbm_ap},
    'CatBoost': {'proba': cat_proba,  'auc': cat_auc,  'ap': cat_ap},
}

comparison_rows = []
for name, info in models_info.items():
    preds = (info['proba'] >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_test, preds)
    f1 = f1_score(y_test, preds)
    recall = cm[1,1] / (cm[1,0] + cm[1,1])
    precision = cm[1,1] / (cm[0,1] + cm[1,1])
    comparison_rows.append({
        'Model': name,
        'ROC-AUC': round(info['auc'], 4),
        'Avg Precision': round(info['ap'], 4),
        'F1 (Disrupted)': round(f1, 4),
        'Recall': round(recall, 4),
        'Precision': round(precision, 4),
        'FP': int(cm[0, 1]),
        'FN': int(cm[1, 0]),
    })

comp_df = pd.DataFrame(comparison_rows)
print(comp_df.to_string(index=False))

---
## Ensemble Methods
We combine the 3 base models using 4 different strategies and compare.

In [ ]:
# Cell 13: Ensemble Method 1 - Simple Average
print('=' * 60)
print('ENSEMBLE 1: Simple Average')
print('=' * 60)

ens_avg_proba = (xgb_proba + lgbm_proba + cat_proba) / 3.0
ens_avg_auc = roc_auc_score(y_test, ens_avg_proba)
ens_avg_ap = average_precision_score(y_test, ens_avg_proba)
ens_avg_preds = (ens_avg_proba >= THRESHOLD).astype(int)

print(f'  ROC-AUC: {ens_avg_auc:.4f}  Avg Precision: {ens_avg_ap:.4f}')
print(f'  F1 (Disrupted): {f1_score(y_test, ens_avg_preds):.4f}')
print()
print(classification_report(y_test, ens_avg_preds, target_names=['Stable','Disrupted']))
cm = confusion_matrix(y_test, ens_avg_preds)
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}  FN={cm[1,0]:,}  TP={cm[1,1]:,}')

In [ ]:
# Cell 14: Ensemble Method 2 - Optimized Weighted Average
print('=' * 60)
print('ENSEMBLE 2: Optimized Weighted Average')
print('=' * 60)

# Grid search over weight combinations (step=0.05)
best_auc_w, best_weights = 0, (1/3, 1/3, 1/3)
step = 0.05
weight_range = np.arange(0.0, 1.0 + step, step)

print('Searching weight combinations...')
for w_xgb in weight_range:
    for w_lgbm in weight_range:
        w_cat = round(1.0 - w_xgb - w_lgbm, 4)
        if w_cat < -0.001 or w_cat > 1.001:
            continue
        w_cat = max(0, w_cat)
        combo = w_xgb * xgb_proba + w_lgbm * lgbm_proba + w_cat * cat_proba
        auc = roc_auc_score(y_test, combo)
        if auc > best_auc_w:
            best_auc_w = auc
            best_weights = (round(w_xgb, 2), round(w_lgbm, 2), round(w_cat, 2))

w_xgb, w_lgbm, w_cat = best_weights
print(f'\n  Best weights: XGBoost={w_xgb}, LightGBM={w_lgbm}, CatBoost={w_cat}')

ens_wt_proba = w_xgb * xgb_proba + w_lgbm * lgbm_proba + w_cat * cat_proba
ens_wt_auc = roc_auc_score(y_test, ens_wt_proba)
ens_wt_ap = average_precision_score(y_test, ens_wt_proba)
ens_wt_preds = (ens_wt_proba >= THRESHOLD).astype(int)

print(f'  ROC-AUC: {ens_wt_auc:.4f}  Avg Precision: {ens_wt_ap:.4f}')
print(f'  F1 (Disrupted): {f1_score(y_test, ens_wt_preds):.4f}')
print()
print(classification_report(y_test, ens_wt_preds, target_names=['Stable','Disrupted']))
cm = confusion_matrix(y_test, ens_wt_preds)
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}  FN={cm[1,0]:,}  TP={cm[1,1]:,}')

In [ ]:
# Cell 15: Ensemble Method 3 - Rank Average
print('=' * 60)
print('ENSEMBLE 3: Rank Average')
print('=' * 60)
print('(Converts probabilities to percentile ranks before averaging')
print(' -> robust to differences in calibration between models)\n')

# Convert each model probabilities to percentile ranks [0, 1]
n = len(y_test)
rank_xgb  = rankdata(xgb_proba)  / n
rank_lgbm = rankdata(lgbm_proba) / n
rank_cat  = rankdata(cat_proba)  / n

ens_rank_proba = (rank_xgb + rank_lgbm + rank_cat) / 3.0
ens_rank_auc = roc_auc_score(y_test, ens_rank_proba)
ens_rank_ap = average_precision_score(y_test, ens_rank_proba)

# For rank average, find the optimal threshold on this scale
best_f1_rank, best_thresh_rank = 0, 0.5
for t in np.arange(0.3, 0.7, 0.01):
    preds_t = (ens_rank_proba >= t).astype(int)
    f1_t = f1_score(y_test, preds_t)
    if f1_t > best_f1_rank:
        best_f1_rank = f1_t
        best_thresh_rank = t

ens_rank_preds = (ens_rank_proba >= best_thresh_rank).astype(int)
print(f'  Optimal threshold (rank scale): {best_thresh_rank:.2f}')
print(f'  ROC-AUC: {ens_rank_auc:.4f}  Avg Precision: {ens_rank_ap:.4f}')
print(f'  F1 (Disrupted): {f1_score(y_test, ens_rank_preds):.4f}')
print()
print(classification_report(y_test, ens_rank_preds, target_names=['Stable','Disrupted']))
cm = confusion_matrix(y_test, ens_rank_preds)
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}  FN={cm[1,0]:,}  TP={cm[1,1]:,}')

In [ ]:
# Cell 16: Ensemble Method 4 - Stacking (Logistic Regression meta-learner)
print('=' * 60)
print('ENSEMBLE 4: Stacking (Logistic Regression Meta-Learner)')
print('=' * 60)
print('Using 5-fold CV on training set to generate meta-features...\n')

# Generate out-of-fold predictions for the training set
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

meta_train = np.zeros((len(X_train), 3))
meta_test = np.zeros((len(X_test), 3))

base_configs = [
    ('XGBoost', lambda: xgb.XGBClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        tree_method='hist', eval_metric='logloss',
        random_state=42, n_jobs=-1)),
    ('LightGBM', lambda: LGBMClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        is_unbalance=True, random_state=42, n_jobs=-1, verbose=-1)),
    ('CatBoost', lambda: CatBoostClassifier(
        iterations=500, depth=8, learning_rate=0.05,
        eval_metric='AUC', auto_class_weights='Balanced',
        random_seed=42, verbose=0)),
]

X_train_np = X_train.values
y_train_np = y_train.values
X_test_np = X_test.values

for i, (name, make_model) in enumerate(base_configs):
    print(f'  Generating OOF predictions for {name}...')
    test_preds_sum = np.zeros(len(X_test))
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_np, y_train_np)):
        print(f'    Fold {fold+1}/{n_folds}', end='... ')
        X_tr, X_val = X_train_np[train_idx], X_train_np[val_idx]
        y_tr, y_val = y_train_np[train_idx], y_train_np[val_idx]

        mdl = make_model()
        if name == 'XGBoost':
            mdl.fit(X_tr, y_tr, verbose=False)
        else:
            mdl.fit(X_tr, y_tr)

        meta_train[val_idx, i] = mdl.predict_proba(X_val)[:, 1]
        test_preds_sum += mdl.predict_proba(X_test_np)[:, 1]
        auc_fold = roc_auc_score(y_val, meta_train[val_idx, i])
        print(f'AUC={auc_fold:.4f}')

    meta_test[:, i] = test_preds_sum / n_folds
    print(f'  {name} OOF AUC: {roc_auc_score(y_train, meta_train[:, i]):.4f}\n')

# Train the meta-learner
print('Training Logistic Regression meta-learner...')
meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_lr.fit(meta_train, y_train)

ens_stack_proba = meta_lr.predict_proba(meta_test)[:, 1]
ens_stack_auc = roc_auc_score(y_test, ens_stack_proba)
ens_stack_ap = average_precision_score(y_test, ens_stack_proba)
ens_stack_preds = (ens_stack_proba >= THRESHOLD).astype(int)

print(f'\n  Meta-learner coefficients: XGB={meta_lr.coef_[0][0]:.4f}, LGBM={meta_lr.coef_[0][1]:.4f}, Cat={meta_lr.coef_[0][2]:.4f}')
print(f'  Meta-learner intercept: {meta_lr.intercept_[0]:.4f}')
print(f'\n  ROC-AUC: {ens_stack_auc:.4f}  Avg Precision: {ens_stack_ap:.4f}')
print(f'  F1 (Disrupted): {f1_score(y_test, ens_stack_preds):.4f}')
print()
print(classification_report(y_test, ens_stack_preds, target_names=['Stable','Disrupted']))
cm = confusion_matrix(y_test, ens_stack_preds)
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}  FN={cm[1,0]:,}  TP={cm[1,1]:,}')

---
## Results Comparison

In [ ]:
# Cell 17: Grand comparison table
print('=' * 60)
print('FULL COMPARISON: Individual vs Ensemble')
print('=' * 60)

all_methods = {
    'XGBoost':            {'proba': xgb_proba,       'auc': xgb_auc,       'ap': xgb_ap},
    'LightGBM':           {'proba': lgbm_proba,      'auc': lgbm_auc,      'ap': lgbm_ap},
    'CatBoost':           {'proba': cat_proba,       'auc': cat_auc,       'ap': cat_ap},
    'Ens: Simple Avg':    {'proba': ens_avg_proba,   'auc': ens_avg_auc,   'ap': ens_avg_ap},
    'Ens: Weighted Avg':  {'proba': ens_wt_proba,    'auc': ens_wt_auc,    'ap': ens_wt_ap},
    'Ens: Rank Avg':      {'proba': ens_rank_proba,  'auc': ens_rank_auc,  'ap': ens_rank_ap},
    'Ens: Stacking (LR)': {'proba': ens_stack_proba, 'auc': ens_stack_auc, 'ap': ens_stack_ap},
}

rows = []
for name, info in all_methods.items():
    p = info['proba']
    if 'Rank' in name:
        preds = (p >= best_thresh_rank).astype(int)
    else:
        preds = (p >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_test, preds)
    rows.append({
        'Method': name,
        'ROC-AUC': round(info['auc'], 4),
        'Avg Prec': round(info['ap'], 4),
        'Recall': round(cm[1,1]/(cm[1,0]+cm[1,1]), 4),
        'Precision': round(cm[1,1]/(cm[0,1]+cm[1,1]), 4),
        'F1': round(f1_score(y_test, preds), 4),
        'FP': int(cm[0,1]),
        'FN': int(cm[1,0]),
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

best_idx = results_df['ROC-AUC'].idxmax()
print(f'\nBest ROC-AUC: {results_df.loc[best_idx, "Method"]} = {results_df.loc[best_idx, "ROC-AUC"]:.4f}')
best_f1_idx = results_df['F1'].idxmax()
print(f'Best F1:      {results_df.loc[best_f1_idx, "Method"]} = {results_df.loc[best_f1_idx, "F1"]:.4f}')
best_fp_idx = results_df['FP'].idxmin()
print(f'Fewest FP:   {results_df.loc[best_fp_idx, "Method"]} = {results_df.loc[best_fp_idx, "FP"]:,}')

In [ ]:
# Cell 18: Visualization - ROC, PR, Bar comparison
colors = {
    'XGBoost': '#e74c3c',
    'LightGBM': '#3498db',
    'CatBoost': '#2ecc71',
    'Ens: Simple Avg': '#9b59b6',
    'Ens: Weighted Avg': '#e67e22',
    'Ens: Rank Avg': '#1abc9c',
    'Ens: Stacking (LR)': '#f39c12',
}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# ROC Curves
for name, info in all_methods.items():
    fpr, tpr, _ = roc_curve(y_test, info['proba'])
    lw = 3 if 'Ens' in name else 1.5
    ls = '-' if 'Ens' in name else '--'
    axes[0,0].plot(fpr, tpr, color=colors[name], lw=lw, ls=ls,
                   label=f"{name} ({info['auc']:.4f})")
axes[0,0].plot([0,1],[0,1],'k:',lw=0.5)
axes[0,0].set_title('ROC Curves', fontsize=14)
axes[0,0].set_xlabel('False Positive Rate')
axes[0,0].set_ylabel('True Positive Rate')
axes[0,0].legend(fontsize=9, loc='lower right')

# PR Curves
for name, info in all_methods.items():
    prec, rec, _ = precision_recall_curve(y_test, info['proba'])
    lw = 3 if 'Ens' in name else 1.5
    ls = '-' if 'Ens' in name else '--'
    axes[0,1].plot(rec, prec, color=colors[name], lw=lw, ls=ls,
                   label=f"{name} ({info['ap']:.4f})")
axes[0,1].set_title('Precision-Recall Curves', fontsize=14)
axes[0,1].set_xlabel('Recall')
axes[0,1].set_ylabel('Precision')
axes[0,1].legend(fontsize=9, loc='lower left')

# AUC Bar Chart
method_names = list(all_methods.keys())
aucs = [all_methods[m]['auc'] for m in method_names]
bar_colors = [colors[m] for m in method_names]
bars = axes[1,0].barh(range(len(method_names)), aucs, color=bar_colors, edgecolor='white')
axes[1,0].set_yticks(range(len(method_names)))
axes[1,0].set_yticklabels(method_names, fontsize=10)
axes[1,0].set_xlim(min(aucs)-0.01, max(aucs)+0.005)
axes[1,0].set_title('ROC-AUC Comparison', fontsize=14)
for bar, auc in zip(bars, aucs):
    axes[1,0].text(auc + 0.0005, bar.get_y() + bar.get_height()/2,
                   f'{auc:.4f}', va='center', fontsize=10, fontweight='bold')
axes[1,0].invert_yaxis()

# Improvement over best individual
best_individual_auc = max(xgb_auc, lgbm_auc, cat_auc)
improvements = {}
for name in ['Ens: Simple Avg', 'Ens: Weighted Avg', 'Ens: Rank Avg', 'Ens: Stacking (LR)']:
    improvements[name] = (all_methods[name]['auc'] - best_individual_auc) * 100

imp_names = list(improvements.keys())
imp_vals = list(improvements.values())
imp_colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in imp_vals]
axes[1,1].barh(range(len(imp_names)), imp_vals, color=imp_colors, edgecolor='white')
axes[1,1].set_yticks(range(len(imp_names)))
axes[1,1].set_yticklabels(imp_names, fontsize=10)
axes[1,1].axvline(0, color='black', lw=0.8)
axes[1,1].set_title(f'AUC Improvement over Best Individual ({best_individual_auc:.4f})', fontsize=14)
axes[1,1].set_xlabel('Delta AUC (percentage points)')
for i, v in enumerate(imp_vals):
    axes[1,1].text(v + 0.01 if v >= 0 else v - 0.01, i, f'{v:+.3f}pp',
                   va='center', fontsize=10, fontweight='bold',
                   ha='left' if v >= 0 else 'right')
axes[1,1].invert_yaxis()

plt.suptitle('CareAssist Ensemble Model Comparison', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 18b: Confusion matrices for all 7 methods
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

for i, (name, info) in enumerate(all_methods.items()):
    if 'Rank' in name:
        preds = (info['proba'] >= best_thresh_rank).astype(int)
    else:
        preds = (info['proba'] >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
                xticklabels=['Stable','Disrupted'],
                yticklabels=['Stable','Disrupted'],
                ax=axes[i])
    axes[i].set_title(f"{name}\nAUC={info['auc']:.4f}", fontsize=11)

axes[-1].axis('off')
plt.suptitle('Confusion Matrices (threshold=0.4)', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 18c: Model agreement analysis
print('=' * 60)
print('MODEL AGREEMENT ANALYSIS')
print('=' * 60)

xgb_pred  = (xgb_proba  >= THRESHOLD).astype(int)
lgbm_pred = (lgbm_proba >= THRESHOLD).astype(int)
cat_pred  = (cat_proba  >= THRESHOLD).astype(int)

agree_all  = ((xgb_pred == lgbm_pred) & (lgbm_pred == cat_pred)).sum()
agree_pct  = 100 * agree_all / len(y_test)
print(f'\nAll 3 models agree: {agree_all:,} / {len(y_test):,} ({agree_pct:.1f}%)')

agree_mask = (xgb_pred == lgbm_pred) & (lgbm_pred == cat_pred)
print(f'Accuracy when all agree: {(y_test[agree_mask] == xgb_pred[agree_mask]).mean():.4f}')
print(f'Accuracy when they disagree: {(y_test[~agree_mask] == ens_avg_preds[~agree_mask]).mean():.4f}')

print('\nProbability correlation matrix:')
corr_df = pd.DataFrame({
    'XGBoost': xgb_proba,
    'LightGBM': lgbm_proba,
    'CatBoost': cat_proba,
}).corr()
print(corr_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_df, annot=True, fmt='.4f', cmap='coolwarm',
            vmin=0.9, vmax=1.0, ax=ax)
ax.set_title('Base Model Probability Correlation')
plt.tight_layout()
plt.savefig('model_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 18d: Feature importance (averaged across all 3 models)
print('=' * 60)
print('ENSEMBLE FEATURE IMPORTANCE (Averaged)')
print('=' * 60)

xgb_imp = xgb_model.feature_importances_ / xgb_model.feature_importances_.sum()
lgbm_imp = lgbm_model.feature_importances_ / lgbm_model.feature_importances_.sum()
cat_imp = cat_model.feature_importances_ / cat_model.feature_importances_.sum()

avg_imp = (xgb_imp + lgbm_imp + cat_imp) / 3.0

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'XGBoost': xgb_imp,
    'LightGBM': lgbm_imp,
    'CatBoost': cat_imp,
    'Ensemble Avg': avg_imp,
}).sort_values('Ensemble Avg', ascending=False)

fig, ax = plt.subplots(figsize=(12, 9))
top15 = feat_imp.head(15)
x = np.arange(len(top15))
w = 0.22
ax.barh(x - 1.5*w, top15['XGBoost'], w, label='XGBoost', color='#e74c3c', alpha=0.8)
ax.barh(x - 0.5*w, top15['LightGBM'], w, label='LightGBM', color='#3498db', alpha=0.8)
ax.barh(x + 0.5*w, top15['CatBoost'], w, label='CatBoost', color='#2ecc71', alpha=0.8)
ax.barh(x + 1.5*w, top15['Ensemble Avg'], w, label='Ensemble Avg', color='#9b59b6', alpha=0.9)
ax.set_yticks(x)
ax.set_yticklabels(top15['feature'])
ax.invert_yaxis()
ax.set_title('Top 15 Features - Importance by Model', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('feature_importance_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 15 features (ensemble average):')
print(feat_imp[['feature','Ensemble Avg','XGBoost','LightGBM','CatBoost']].head(15).to_string(index=False))

In [ ]:
# Cell 18e: SHAP on XGBoost (representative base model)
print('Computing SHAP on XGBoost (sample=1000)...')
explainer = shap.TreeExplainer(xgb_model)
X_sample = X_test.sample(min(1000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False, max_display=20)
plt.title('SHAP Feature Impact (XGBoost base model)')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

idx = np.argmax(xgb_model.predict_proba(X_sample)[:,1])
ev = explainer.expected_value
if isinstance(ev, list): ev = ev[1]
shap.plots.waterfall(shap.Explanation(
    values=shap_values[idx], base_values=ev,
    data=X_sample.iloc[idx], feature_names=feature_names), show=True)
print('SHAP done')

In [ ]:
# Cell 19: Score all records with weighted ensemble
print('Scoring all records with weighted ensemble...')
X_all = model_df.drop(columns=['disruption'])

xgb_all  = xgb_model.predict_proba(X_all)[:, 1]
lgbm_all = lgbm_model.predict_proba(X_all)[:, 1]
cat_all  = cat_model.predict_proba(X_all)[:, 1]

df['xgb_score']  = xgb_all
df['lgbm_score'] = lgbm_all
df['cat_score']  = cat_all
df['ensemble_score'] = w_xgb * xgb_all + w_lgbm * lgbm_all + w_cat * cat_all

df['risk_tier'] = pd.cut(df['ensemble_score'],
    bins=[0, 0.3, 0.6, 0.8, 1.0],
    labels=['Low', 'Medium', 'High', 'Critical'],
    include_lowest=True)

print('\nRisk Tier Distribution (Weighted Ensemble):')
tier_dist = df['risk_tier'].value_counts()
for t in ['Critical', 'High', 'Medium', 'Low']:
    if t in tier_dist.index:
        print(f'  {t:>10}: {tier_dist[t]:>10,} ({100*tier_dist[t]/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['ensemble_score'], bins=50, color='#9b59b6', edgecolor='white', alpha=0.8)
for v, c, l in [(0.3,'green','Low/Med'),(0.6,'orange','Med/High'),(0.8,'red','High/Crit')]:
    ax.axvline(v, color=c, ls='--', lw=2, label=l)
ax.set_title('Ensemble Priority Score Distribution', fontsize=14)
ax.set_xlabel('Ensemble Score')
ax.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 20: Export everything
os.makedirs('careassist_ensemble_output', exist_ok=True)

joblib.dump(xgb_model,  'careassist_ensemble_output/xgboost_model.pkl')
joblib.dump(lgbm_model, 'careassist_ensemble_output/lightgbm_model.pkl')
joblib.dump(cat_model,  'careassist_ensemble_output/catboost_model.pkl')
joblib.dump(meta_lr,    'careassist_ensemble_output/meta_learner_lr.pkl')
print('Saved: xgboost_model.pkl, lightgbm_model.pkl, catboost_model.pkl, meta_learner_lr.pkl')

meta = {
    'approach': 'Ensemble (XGBoost + LightGBM + CatBoost)',
    'individual_aucs': {
        'XGBoost': round(xgb_auc, 4),
        'LightGBM': round(lgbm_auc, 4),
        'CatBoost': round(cat_auc, 4),
    },
    'ensemble_aucs': {
        'simple_average': round(ens_avg_auc, 4),
        'weighted_average': round(ens_wt_auc, 4),
        'rank_average': round(ens_rank_auc, 4),
        'stacking_lr': round(ens_stack_auc, 4),
    },
    'weighted_avg_weights': {
        'XGBoost': w_xgb,
        'LightGBM': w_lgbm,
        'CatBoost': w_cat,
    },
    'threshold': THRESHOLD,
    'feature_names': feature_names,
    'feature_count': len(feature_names),
    'train_samples': int(len(X_train)),
    'test_samples': int(len(X_test)),
    'total_records': int(len(df)),
    'disruption_rate': round(float(y.mean()), 4),
    'years': sorted([str(x) for x in df['FY'].dropna().unique()]),
}
with open('careassist_ensemble_output/ensemble_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved: ensemble_metadata.json')

results_df.to_csv('careassist_ensemble_output/comparison_results.csv', index=False)
feat_imp.to_csv('careassist_ensemble_output/feature_importance.csv', index=False)
print('Saved: comparison_results.csv, feature_importance.csv')

export_cols = [
    'RecNumbr','StFCID','FIPSCode','FY',
    'age_at_removal','total_removals','placement_type','case_goal',
    'has_disability','has_clinical_disability','has_behavioral','num_removal_reasons',
    'los_current_setting','los_latest_removal','disruption',
    'xgb_score','lgbm_score','cat_score','ensemble_score','risk_tier'
]
export_cols = [c for c in export_cols if c in df.columns]
df[export_cols].to_csv('careassist_ensemble_output/scored_cases.csv', index=False)
print(f'Saved: scored_cases.csv ({len(df):,} records)')

for fn in ['eda_plots.png', 'ensemble_comparison.png', 'confusion_matrices.png',
           'model_correlation.png', 'feature_importance_ensemble.png',
           'shap_summary.png', 'score_distribution.png']:
    if os.path.exists(fn):
        shutil.copy(fn, f'careassist_ensemble_output/{fn}')

all_files = os.listdir('careassist_ensemble_output')
print(f'\nAll output files ({len(all_files)}): {all_files}')

In [ ]:
# Cell 21: Download zip
shutil.make_archive('careassist_ensemble_output', 'zip', '.', 'careassist_ensemble_output')
print('Downloading careassist_ensemble_output.zip...')
files.download('careassist_ensemble_output.zip')

In [ ]:
# Cell 22: Final Summary
print('=' * 60)
print('CAREASSIST ENSEMBLE MODEL - FINAL SUMMARY')
print('=' * 60)

print(f'\nDATA: {len(df):,} records, FY {sorted(df["FY"].dropna().unique())}')
print(f'FEATURES: {len(feature_names)} (race, gender, NUMPLEP excluded)')
print(f'DISRUPTION RATE: {y.mean():.1%}')
print(f'TRAIN: {len(X_train):,}  TEST: {len(X_test):,}')

print(f'\n--- INDIVIDUAL MODELS ---')
print(f'  XGBoost  ROC-AUC: {xgb_auc:.4f}  AP: {xgb_ap:.4f}')
print(f'  LightGBM ROC-AUC: {lgbm_auc:.4f}  AP: {lgbm_ap:.4f}')
print(f'  CatBoost ROC-AUC: {cat_auc:.4f}  AP: {cat_ap:.4f}')

print(f'\n--- ENSEMBLE METHODS ---')
print(f'  Simple Average   ROC-AUC: {ens_avg_auc:.4f}  AP: {ens_avg_ap:.4f}')
print(f'  Weighted Average ROC-AUC: {ens_wt_auc:.4f}  AP: {ens_wt_ap:.4f}  (w={best_weights})')
print(f'  Rank Average     ROC-AUC: {ens_rank_auc:.4f}  AP: {ens_rank_ap:.4f}')
print(f'  Stacking (LR)    ROC-AUC: {ens_stack_auc:.4f}  AP: {ens_stack_ap:.4f}')

best_ens_name = max(
    [('Simple Average', ens_avg_auc), ('Weighted Average', ens_wt_auc),
     ('Rank Average', ens_rank_auc), ('Stacking (LR)', ens_stack_auc)],
    key=lambda x: x[1]
)
best_ind_auc = max(xgb_auc, lgbm_auc, cat_auc)
improvement = (best_ens_name[1] - best_ind_auc) * 100

print(f'\n--- RESULT ---')
print(f'  Best individual: {best_ind_auc:.4f}')
print(f'  Best ensemble:   {best_ens_name[0]} = {best_ens_name[1]:.4f}')
print(f'  Improvement:     {improvement:+.2f} percentage points')

print(f'\n--- RISK TIER DISTRIBUTION ---')
for t in ['Critical', 'High', 'Medium', 'Low']:
    if t in tier_dist.index:
        print(f'  {t:>10}: {tier_dist[t]:>10,} ({100*tier_dist[t]/len(df):.1f}%)')

print(f'\n--- TOP 10 FEATURES (Ensemble Average) ---')
for _, r in feat_imp.head(10).iterrows():
    print(f'  {r["feature"]:>30s}: {r["Ensemble Avg"]:.4f}')

print('\n' + '=' * 60)
print('Done! All outputs saved to careassist_ensemble_output.zip')
print('=' * 60)